In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import GlobalMaxPooling2D
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
import numpy as np
from numpy.linalg import norm
import os
import pickle

In [ ]:
path = "images"
os.listdir(path)

['31973.jpg',
 '30778.jpg',
 '19812.jpg',
 '22735.jpg',
 '38246.jpg',
 '16916.jpg',
 '52876.jpg',
 '39500.jpg',
 '44758.jpg',
 '59454.jpg',
 '44951.jpg',
 '34410.jpg',
 '22706.jpg',
 '59735.jpg',
 '12666.jpg',
 '13288.jpg',
 '36417.jpg',
 '6234.jpg',
 '36765.jpg',
 '38745.jpg',
 '28578.jpg',
 '50478.jpg',
 '52446.jpg',
 '48069.jpg',
 '24819.jpg',
 '47378.jpg',
 '3863.jpg',
 '6241.jpg',
 '59221.jpg',
 '25540.jpg',
 '50446.jpg',
 '57177.jpg',
 '45816.jpg',
 '20529.jpg',
 '46250.jpg',
 '10304.jpg',
 '27582.jpg',
 '2193.jpg',
 '11925.jpg',
 '21440.jpg',
 '56540.jpg',
 '3750.jpg',
 '46634.jpg',
 '11378.jpg',
 '2008.jpg',
 '10730.jpg',
 '52925.jpg',
 '41022.jpg',
 '41921.jpg',
 '40543.jpg',
 '22608.jpg',
 '16730.jpg',
 '33135.jpg',
 '42643.jpg',
 '5982.jpg',
 '33119.jpg',
 '34948.jpg',
 '30768.jpg',
 '7737.jpg',
 '26556.jpg',
 '32573.jpg',
 '47655.jpg',
 '26645.jpg',
 '14986.jpg',
 '35303.jpg',
 '10597.jpg',
 '37867.jpg',
 '34283.jpg',
 '57160.jpg',
 '19299.jpg',
 '55316.jpg',
 '6588.jpg',
 

In [ ]:
filenames = sorted([os.path.join(path, f) for f in os.listdir(path)])
print(f"Total images: {len(filenames)}")
filenames

Total images: 44441


['/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10000.jpg',
 '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10001.jpg',
 '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10002.jpg',
 '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10003.jpg',
 '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10004.jpg',
 '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10005.jpg',
 '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10006.jpg',
 '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10007.jpg',
 '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10008.jpg',
 '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10009.jpg',
 '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10010.jpg',
 '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/10011.jpg',
 '/kaggle/input/

In [ ]:
# Used as feature extractor
# Will produce 2048 features of each image
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    GlobalMaxPooling2D()
])

model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d            │ (None, 2048)           │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
BATCH_SIZE = 128 

def load_and_preprocess(filepath):
    img_bytes = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, (224, 224))
    img = preprocess_input(img)
    return img

In [ ]:
ds = tf.data.Dataset.from_tensor_slices(filenames)
ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
ds = ds.batch(BATCH_SIZE)
ds = ds.prefetch(tf.data.AUTOTUNE)  # overlaps CPU preprocessing with GPU inference

In [20]:
feature_list = model.predict(ds, verbose=1)

348/348 ━━━━━━━━━━━━━━━━━━━━ 714s 2s/step


In [ ]:
# Normalize each row
norms = np.linalg.norm(feature_list, axis=1, keepdims=True)
feature_list = feature_list / norms

In [ ]:
np.save('embeddings.npy', feature_list)
pickle.dump(filenames, open('filenames.pkl', 'wb'))
model.save('feature_extractor.keras') 